# Singular Value Decomposition (SVD)

This notebook implements a Matrix Factorization recommendation system using Singular Value Decomposition (SVD) on the Netflix Prize dataset.
It covers data preparation, train/test splitting, model training using the Surprise library, rating prediction, RMSE evaluation, MAP@10 evaluation, Top-K recommendation generation, and recommendation quality analysis.

## Load Data

In [1]:
import pandas as pd
import numpy as np

filtered_df = pd.read_csv(
    "../data/processed/filtered_netflix.csv"
)

print(filtered_df.shape)
filtered_df.head()

(4660641, 3)


,user_id,movie_id,rating
0,1488844,1,3
1,885013,1,4
2,30878,1,4
3,823519,1,3
4,893988,1,3


In [2]:
movies = pd.read_csv(
    "../data/movie_titles.csv",
    header=None,
    encoding="latin1",
    engine="python",
    names=["movie_id", "year", "title"],
    on_bad_lines="skip"
)
print("Movies loaded:", movies.shape)
movies.head()

Movies loaded: (17434, 3)


,movie_id,year,title
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review
2,3,1997.0,Character
3,4,1994.0,Paula Abdul's Get Up & Dance
4,5,2004.0,The Rise and Fall of ECW


In [3]:
from surprise import Dataset, Reader, SVD
print("Success")

Success


In [4]:
svd_df = filtered_df.sample(
    1000000,
    random_state=42
)
print(svd_df.shape)

(1000000, 3)


In [5]:

reader = Reader(rating_scale=(1, 5))

data = Dataset.load_from_df(
    svd_df[['user_id', 'movie_id', 'rating']],
    reader
)

In [6]:
from surprise.model_selection import train_test_split

trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

In [7]:
import time
from surprise import SVD

start = time.time()

model = SVD(
    n_factors=20,
    n_epochs=30,
    reg_all=0.05,
    random_state=42
)
model.fit(trainset)

end = time.time()

print(
    f"Training Time: {end-start:.2f} seconds"
)

Training Time: 9.13 seconds


In [8]:
train_df_svd = pd.DataFrame(
    [
        (
            trainset.to_raw_uid(uid),
            trainset.to_raw_iid(iid),
            rating
        )
        for (uid, iid, rating)
        in trainset.all_ratings()
    ],
    columns=[
        "user_id",
        "movie_id",
        "rating"
    ]
)

train_df_svd["user_id"] = train_df_svd["user_id"].astype(int)
train_df_svd["movie_id"] = train_df_svd["movie_id"].astype(int)

## RMSE Evaluation

In [9]:
from surprise import accuracy

predictions = model.test(testset)

rmse = accuracy.rmse(
    predictions,
    verbose=True
)

RMSE: 0.9712


## MAP@10 Evaluation

In [10]:
all_movies = set(
    svd_df["movie_id"].unique()
)

def recommend_svd(user_id, top_n=10):

    watched = set(
        train_df_svd[
            train_df_svd["user_id"] == user_id
        ]["movie_id"]
    )

    candidates = all_movies - watched

    predictions = []

    for movie in candidates:

        pred = model.predict(
            user_id,
            movie
        )

        predictions.append(
            (movie, pred.est)
        )

    predictions.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return predictions[:top_n]

In [11]:
sample_user = svd_df.iloc[0]['user_id']
recommend_svd(sample_user)

[(np.int64(13), 5),
 (np.int64(32), 5),
 (np.int64(33), 5),
 (np.int64(37), 5),
 (np.int64(68), 5),
 (np.int64(73), 5),
 (np.int64(76), 5),
 (np.int64(85), 5),
 (np.int64(103), 5),
 (np.int64(106), 5)]

In [12]:
def average_precision_at_k(
    recommended,
    relevant,
    k=10
):

    score = 0
    hits = 0

    for i, movie in enumerate(
        recommended[:k],
        start=1
    ):

        if movie in relevant:

            hits += 1
            score += hits / i

    if len(relevant) == 0:
        return 0

    return score / min(
        len(relevant),
        k
    )

In [13]:
test_df_svd = pd.DataFrame(
    testset,
    columns=[
        'user_id',
        'movie_id',
        'rating'
    ]
)

test_df_svd.head()

,user_id,movie_id,rating
0,1825376,290,3.0
1,755820,789,4.0
2,841974,187,3.0
3,77584,189,3.0
4,2250613,460,2.0


In [14]:
sample_users = (
    test_df_svd['user_id']
    .drop_duplicates()
    .sample(
        100,
        random_state=42
    )
)

In [15]:
ap_scores = []

for user in sample_users:

    user_test = test_df_svd[
        test_df_svd['user_id'] == user
    ]

    relevant_movies = set(
        user_test[
            user_test['rating'] >= 4
        ]['movie_id']
    )

    if len(relevant_movies) == 0:
        continue

    recs = recommend_svd(
        user,
        top_n=10
    )

    recommended_movies = [
        movie
        for movie, _
        in recs
    ]

    ap = average_precision_at_k(
        recommended_movies,
        relevant_movies,
        k=10
    )
    ap_scores.append(ap)

In [16]:
import numpy as np
map10 = np.mean(ap_scores)

print("SVD MAP@10:",map10)

SVD MAP@10: 0.0


### Analysis of SVD Results

The SVD model achieved the best RMSE among the evaluated approaches, indicating strong rating prediction accuracy. However, its MAP@10 score was close to zero, meaning that highly relevant movies rarely appeared in the top-ranked recommendations.

This highlights an important distinction between rating prediction and recommendation ranking. While SVD successfully learns latent user-item interactions, optimizing for RMSE does not necessarily produce effective Top-K recommendations. In contrast, Item-Based Collaborative Filtering achieved a slightly worse RMSE but substantially better recommendation ranking performance.

Therefore, for personalized content discovery, Item-Based Collaborative Filtering proved more effective despite its lower rating prediction accuracy.


In [17]:
user = sample_users.iloc[0]

relevant_movies = set(
    test_df_svd[
        (test_df_svd['user_id'] == user)
        &
        (test_df_svd['rating'] >= 4)
    ]['movie_id']
)

recs = recommend_svd(user, top_n=10)
recs_df = pd.DataFrame(
    recs,
    columns=[
        'movie_id',
        'predicted_rating'
    ]
)

recs_df = recs_df.merge(
    movies,
    on='movie_id',
    how='left'
)

display(recs_df)
recommended_movies = [
    movie
    for movie, _
    in recs
]

overlap = relevant_movies.intersection(
    recommended_movies
)

print("=" * 50)
print(f"USER: {user}")
print("=" * 50)

print("\nRelevant Movies:")
print(list(relevant_movies))

print("\nRecommended Movies:")
print(recommended_movies)

print("\nOverlap:")
print(overlap)

print("\nAP@10:")
print(
    average_precision_at_k(
        recommended_movies,
        relevant_movies,
        k=10
    )
)

,movie_id,predicted_rating,year,title
0,85,4.390908,2005.0,Elfen Lied
1,463,4.327389,1962.0,The Twilight Zone: Vol. 12
2,13,4.326531,2003.0,Lord of the Rings: The Return of the King: Ext...
3,575,4.282243,1994.0,Highlander: Season 4
4,135,4.258887,1998.0,GTO: Great Teacher Onizuka: Set 2
5,270,4.247190,2001.0,Sex and the City: Season 4
6,316,4.205747,1999.0,Futurama: Monster Robot Maniac Fun Collection
7,908,4.196061,1981.0,Sense and Sensibility
8,595,4.195370,2001.0,Monarch of the Glen: Series 2
9,325,4.170376,2004.0,Ghosts of Rwanda: Frontline


USER: 1747125

Relevant Movies:
[143]

Recommended Movies:
[np.int64(85), np.int64(463), np.int64(13), np.int64(575), np.int64(135), np.int64(270), np.int64(316), np.int64(908), np.int64(595), np.int64(325)]

Overlap:
set()

AP@10:
0.0
